## The State of Tax Justice: Estimate misalignment for 2016

- Author: Mario Cuenda García, based on Alison Schultz, based on Javier Garcia Bernado's work
- Created: 4 November 2024
- Last updated:

**Description**
- This notebook is the third out of three notebooks to estimate the tax losses caused by profit shifting by multinational enterprises (MNEs). The analysis used the misalignment method based on the country-by-country reports (CbCR) published by the OECD.
    - Details on the misalignment method and its background can be found here: https://www.sciencedirect.com/science/article/pii/S0305750X23003455. 
    - The working paper version is here: https://www.econstor.eu/bitstream/10419/286362/1/wp-2023-33.pdf 

- This notebook estimates profit misalignment based on different formulas. It uses the dataset **"data/final/cbcr_main.csv"** (for the estimation with imputed values) or the dataset **"data/final/cbcr_main_noimputation_allsubgroupsonly.csv"** (for the estimation without imputed values). 

**Outline**
1. Define misalignment
2. Calculate misalignment for sample with full information.
3. Calculate misalignment for samples with imputed data and aggregate results. 

**To dos before running this notebook**
- Run the notebooks 1_clean and 2_impute_missings. Note the requirements and instructions given in these notebooks.

**To dos in this notebook**

Change the formula to any required formula in each section and adapt the output name of the csvs. The formulas I have used are like follows, where sales refer to unrelated party revenues and assets to tangible assets excluding cash.

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]

- Adjust the input path in 5.1 to the bootstrapped sample you use

## 0. Load packages

In [1]:
# Packages
import pandas as pd
import numpy as np
import tjn_tools
from config import *
import os

# Show columns and select data format
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.0f}'.format

[TJN TOOLS: Data processing] Module loaded.
[TJN TOOLS: Other functions] Module loaded.
[TJN TOOLS: Paths] Module loaded. Sharepoint FOUND at C:\Users\aliso\Tax Justice Network Ltd


## Step 1. Generate the template datasets

### Step 1.1. Generate the dataset with Unique ISO parents

In [2]:
# Open the original dataset
iso_parents = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Keep just the following columns: iso_parent and year
iso_parents = iso_parents[['iso_parent', 'year']]

# Keep every unique combination of iso_parent and year
iso_parents = iso_parents.drop_duplicates(subset=['iso_parent', 'year'])

# Sort by year, then iso_parent
iso_parents = iso_parents.sort_values(by=['year', 'iso_parent'])

# Filter by year
iso_parents_2016= iso_parents[iso_parents['year'] == 2016]
iso_parents_2017= iso_parents[iso_parents['year'] == 2017]
iso_parents_2018= iso_parents[iso_parents['year'] == 2018]
iso_parents_2019= iso_parents[iso_parents['year'] == 2019]
iso_parents_2020= iso_parents[iso_parents['year'] == 2020]
iso_parents_2021= iso_parents[iso_parents['year'] == 2021]

# OPTIONAL: Print the count of how many unique iso_partner values there are
print(iso_parents_2016['iso_parent'].nunique())
print(iso_parents_2017['iso_parent'].nunique())
print(iso_parents_2018['iso_parent'].nunique())
print(iso_parents_2019['iso_parent'].nunique())
print(iso_parents_2020['iso_parent'].nunique())
print(iso_parents_2021['iso_parent'].nunique())

iso_parents_2016

26
38
46
50
52
52


,iso_parent,year
264,AUS,2016
760,AUT,2016
833,BEL,2016
1063,BMU,2016
1644,BRA,2016
1893,CAN,2016
2722,CHL,2016
2816,CHN,2016
4968,DNK,2016
6205,FIN,2016


### Step 1.2. Generate the dataset with unique iso_partners

In [3]:
## Download the relevant dataset
iso_partners = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Exclude country groups
iso_partners = iso_partners[~iso_partners['iso_partner'].isin(non_countries)]

# Keep just the following columns: iso_partner and year
iso_partners = iso_partners[['iso_partner', 'year']]

# Sort by year, then iso_partner
iso_partners = iso_partners.sort_values(by=['year', 'iso_partner'])

# Keep every unique combination of iso_partner and year
iso_partners = iso_partners.drop_duplicates(subset=['iso_partner', 'year'])

# Filter by year
iso_partners_2016= iso_partners[iso_partners['year'] == 2016]
iso_partners_2017= iso_partners[iso_partners['year'] == 2017]
iso_partners_2018= iso_partners[iso_partners['year'] == 2018]
iso_partners_2019= iso_partners[iso_partners['year'] == 2019]
iso_partners_2020= iso_partners[iso_partners['year'] == 2020]
iso_partners_2021= iso_partners[iso_partners['year'] == 2021]


# Optional: Print the count of how many unique iso_partner values there are 
print(iso_partners_2016['iso_partner'].nunique())
print(iso_partners_2017['iso_partner'].nunique())
print(iso_partners_2018['iso_partner'].nunique())
print(iso_partners_2019['iso_partner'].nunique())
print(iso_partners_2020['iso_partner'].nunique())
print(iso_partners_2021['iso_partner'].nunique())

183
215
213
210
212
211


### Step 1.3. Generate the template dataset

In [4]:
# Perform a cross join to merge all values of iso_partners_ with each value of iso_parents
iso_combinations_2016 = iso_parents_2016.assign(key=1).merge(iso_partners_2016.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2017 = iso_parents_2017.assign(key=1).merge(iso_partners_2017.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2018 = iso_parents_2018.assign(key=1).merge(iso_partners_2018.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2019 = iso_parents_2019.assign(key=1).merge(iso_partners_2019.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2020 = iso_parents_2020.assign(key=1).merge(iso_partners_2020.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2021 = iso_parents_2021.assign(key=1).merge(iso_partners_2021.assign(key=1), on='key').drop('key', axis=1)

# Concatenate all the years
template_dataset = pd.concat([iso_combinations_2016, iso_combinations_2017, iso_combinations_2018, iso_combinations_2019, iso_combinations_2020, iso_combinations_2021])

# Drop year_y
template_dataset = template_dataset.drop(columns=['year_y'])
# Rename year_x to year
template_dataset = template_dataset.rename(columns={'year_x': 'year'})
# Order columns by iso_parent then iso_partner then year
template_dataset = template_dataset[['iso_parent', 'iso_partner', 'year']]
# Generate new column called cbcr_estimates
template_dataset['cbcr_estimates'] = np.nan

template_dataset

,iso_parent,iso_partner,year,cbcr_estimates
0,AUS,ABW,2016,NaN
1,AUS,AFG,2016,NaN
2,AUS,AGO,2016,NaN
3,AUS,ALB,2016,NaN
4,AUS,AND,2016,NaN
...,...,...,...,...
10967,ZAF,XKV,2021,NaN
10968,ZAF,YEM,2021,NaN
10969,ZAF,ZAF,2021,NaN
10970,ZAF,ZMB,2021,NaN


## Step 2. Define the misalignment formula

In [5]:
def calculate_misalignment(cbcr_data,
                           formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",
                                         'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip'],
                           weights=[.5, 0, 0, .5, 0, 0, 0, 0],
                           profit_var='profit_loss_before_income_tax_corrected',
                           etr_max=0.15): 

    # Create variable with positive profits only for calculating shares
    cbcr_data['profit_var_pos'] = cbcr_data[profit_var]
    cbcr_data.loc[cbcr_data[profit_var] < 0, 'profit_var_pos'] = 0
    cbcr_data['share_profit'] = cbcr_data['profit_var_pos'] / cbcr_data.groupby('iso_parent')['profit_var_pos'].transform('sum')

    # Calculate weighted shares of economic activity
    actual_weights = []
    actual_variables = []
    for i, var in enumerate(formula_vars):
        if var is not None and weights[i] > 0:
            actual_variables.append(f"share_{var}")
            actual_weights.append(weights[i])
            cbcr_data.loc[cbcr_data[var] < 0, var] = 0  # Set economic activity measure to zero if negative
            cbcr_data[f"share_{var}"] = cbcr_data[var] / cbcr_data.groupby('iso_parent')[var].transform('sum')

    # Calculate the share of economic activity
    cbcr_data["share_economy_partner_of_parent"] = (cbcr_data.loc[:, actual_variables] * actual_weights).sum(1, min_count=len(actual_weights))
    # Set economic activity to 1% for those jurisdictions without economic activity but with reported profits
    cbcr_data.loc[(cbcr_data["share_economy_partner_of_parent"] == 0) & (cbcr_data[profit_var] > 0), "share_economy_partner_of_parent"] = 0.01

    # Normalize the economic activity shares to sum to 1
    cbcr_data["share_economy_partner_of_parent"] = cbcr_data["share_economy_partner_of_parent"] / cbcr_data.groupby('iso_parent')["share_economy_partner_of_parent"].transform('sum')

    # Calculate theoretical profit and misaligned profit
    cbcr_data["theoretical_profit"] = cbcr_data["share_economy_partner_of_parent"] * cbcr_data.groupby('iso_parent')[profit_var].transform('sum')
    cbcr_data["misaligned_profit"] = cbcr_data[profit_var] - cbcr_data["theoretical_profit"]

    # Set positive misaligned profits to 0 if ETR exceeds the threshold (etr_max)
    cbcr_data.loc[((cbcr_data["misaligned_profit"] > 0) & (cbcr_data["etr_average_corrected"] > etr_max)), "misaligned_profit"] = 0

    # Adjust misalignment per 'iso_parent'
    def adjust_misalignment(group):
        total_negative_misalignment = group.loc[group["misaligned_profit"] < 0, "misaligned_profit"].sum()
        total_positive_misalignment = group.loc[group["misaligned_profit"] > 0, "misaligned_profit"].sum()
        
        # Adjust negative misalignments to balance positive misalignments within each 'iso_parent'
        if total_negative_misalignment != 0:
            factor = - total_positive_misalignment / total_negative_misalignment
            group.loc[group["misaligned_profit"] < 0, "misaligned_profit"] *= factor
        
        return group

    # Apply the adjustment by grouping by 'iso_parent'
    cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)

    return cbcr_data

## Step 3. Calculate misalignment for sample with full information

### Step 3.1 Import data
- Import data without imputed values. This is only the data from the sample of reporting countries that actually is reported on a country basis, i.e. excluding aggregated country groups and data from reporting countries that do not report on a country by country basis, but just by continents.

In [6]:
cbcr_sample = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Exclude country groups
cbcr_sample = cbcr_sample[~cbcr_sample['iso_partner'].isin(non_countries)]

### Step 3.2 Exclude countries that do not report truly country-by-country


- In the 2024 data, the following reporting countries do not report country-by-country. We exclude those from the "clean" analysis where we only use values that are actually in the data.
    - Austria: Only continents in all years
    - Czechia: Only Czechia versus rest of the world from 2019 to 2021
    - Finland: Only Finland and rest of the world between 2016 and 2018 and Finland and continents between 2019 and 2021
    - Greece: Only Greece and continents between 2017 and 2019
    - Hungary: Only Hungary versus rest of the world between 2018 and 2021
    - Isle of Man: Only continents between 2017 and 2020
    - Ireland: Only Ireland versus rest of the world in all years
    - Korea: Only Korea and rest of the world betweem 2016 and 2018 and Korea and continents between 2019 and 2021
    - Macau: Only Macau versus rest of the world between 2019 and 2021
    - Mauritius: Only Mauritius and continents between 2019 and 2021
    - Morocco: Only Morocco and continents in 2021
    - Netherlands: Only Netherlands versus rest of the world between 2016 and 2017
    - Norway: Only Norway and continents 2016 and 2017
    - New Zealand: Only New Zealand versus rest of the world between 2018 and 2021
    - Poland: Only Poland and continents 2019 to 2021
    - Sweden: Only Sweden and continents in all years
    - United Kingdom: Only UK and continents between 2017 and 2021

In [7]:
# Define the conditions for exclusion
exclusion_conditions = [
    ('AUT', 2016, 2021),                # Austria: all years
    ('CZE', 2019, 2021),                # Czechia: from 2019 to 2021
    ('FIN', 2016, 2021),                # Finland: all years
    ('GRC', 2017, 2019),                # Greece: between 2017 and 2019
    ('HUN', 2018, 2021),                # Hungary: between 2018 and 2021
    ('IMN', 2017, 2020),                # Isle of Man: between 2017 and 2020
    ('IRL', 2016, 2021),                # Ireland: all years
    ('KOR', 2016, 2021),                # Korea: all years
    ('MAC', 2019, 2021),                # Macau: between 2019 and 2021
    ('MUS', 2019, 2021),                # Mauritius: between 2019 and 2021
    ('MAR', 2021, 2021),                # Morocco: 2021
    ('NLD', 2016, 2017),                # Netherlands: between 2016 and 2017
    ('NOR', 2016, 2017),                # Norway: 2016 and 2017
    ('NZL', 2018, 2021),                # New Zealand: between 2018 and 2021
    ('POL', 2019, 2021),                # Poland: 2019 to 2021
    ('SWE', 2016, 2021),                # Sweden: all years
    ('GBR', 2017, 2021)                 # United Kingdom: between 2017 and 2021
]

# Iterate through the exclusion conditions
for iso_parent, start_year, end_year in exclusion_conditions:
    cbcr_sample = cbcr_sample[~((cbcr_sample['iso_parent'] == iso_parent) & 
                                     (cbcr_sample['year'].between(start_year, end_year)))]

### Step 3.3 Calculate misalignment for sample countries with full information

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]


The end of this cell shows, among other things, the misaligned profits and the theoretical profits, as well as the CBCR variables.

**More importantly, if for whatever reason we wanted to just run this cell without the "bad reporters", we could just modify the cell to run like the cell from Step 5.2. in order to obtain a dataset of profit shifting without the "bad reporters". In a way, this cell is not really necessary for the rest of the notebook. It just shows an intermediary steps if we want to calculate the misalignment for the "good reporters" only.**

In [8]:
misalignment_2016 = cbcr_sample[cbcr_sample['year'] == 2016].copy()
misalignment_2016 = calculate_misalignment(misalignment_2016, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Keep only the first occurrence of these unique variables for each 'iso_partner'
unique_columns = misalignment_2016.drop_duplicates(subset=['iso_partner'])[['iso_partner', 'partner_jurisdiction', 
                                                                               'etr_average_corrected', 'cit',
                                                                               'tax_revenue_current_usd', 
                                                                               'gvt_health_expenditure', 'region_tjn', 
                                                                               'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

# Keep iso_parent, iso_partner, year, misaligned_profit, theoretical_profit, profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
misalignment_2016 = misalignment_2016[['iso_parent', 'iso_partner', 'year', 'misaligned_profit', 'theoretical_profit', 'profit_loss_before_income_tax_corrected', 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']]

# Show if iso_partner = USA
misalignment_2016[misalignment_2016['iso_parent'] == 'USA']

C:\Users\aliso\AppData\Local\Temp\ipykernel_18052\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip
868,USA,ABW,2016,0,"16,065,071","163,855,057",805,"360,224,979","125,946,296","23,058,933","1,606,215,229","611,512,932","251,287,953",NaN
869,USA,AGO,2016,"-488,760,628","121,080,907","-477,367,266","10,297","1,150,205,281","17,038,207,917","22,798,423","3,154,368,161","5,987,970,691","4,837,765,410",NaN
870,USA,ALB,2016,"-1,604,149","4,336,428","2,372,277",347,"20,775,779","15,881,193","1,594,004","2,078,530","23,890,890","3,115,111",NaN
871,USA,ARE,2016,0,"844,059,293","2,792,816,388","38,222","19,339,638,719","11,203,581,506","1,356,904,444","5,702,338,104","28,633,908,563","9,294,269,844",NaN
872,USA,ARG,2016,0,"1,400,110,950","2,305,131,413","104,461","24,958,390,377","10,002,756,141","785,090,643","13,701,763,791","33,395,306,816","8,436,916,440",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1000,USA,VIR,2016,"-13,047,285","33,535,421","17,560,067","1,538","422,526,798","493,611,386","53,218,896","1,080,481,515","461,725,718","39,198,920",NaN
1001,USA,VNM,2016,"-35,059,090","593,040,590","550,113,547","49,428","5,200,622,455","2,178,540,324","147,559,781","2,860,303,547","7,037,112,654","1,836,490,199",NaN
1002,USA,ZAF,2016,"-180,007,805","1,254,646,632","1,034,241,517","102,349","23,237,577,897","6,592,854,411","391,490,145","11,999,501,483","29,047,941,327","5,810,363,430",NaN
1003,USA,ZMB,2016,"-66,031,576","30,079,153","-50,771,214","2,514","342,266,764","108,339,455","7,234,357","124,036,778","523,769,287","181,502,523",NaN


## Step 4. Generate the dataset for the "bad reporters"

For the "bad" reporters, we are going to assume that their MNEs behave like the "average" MNE in the countries that report correctly. To do that we need to:

1. First aggregate the variables reported by the CBCR by partner countries. For instance, we see that on aggregate, there are 80m employees reported.
2. Then we look at the share corresponding by partners. For instance, 18m employees are reported in the USA. how many does the USA have. In short, roughly 24% of all employees reported are in the USA.
3. We then assume that 24% of the toal employees reported by the "bad" reporters are assigned to the US. And we repeat with all of them



### Step 4.1. Generate the Total Sums of Variables, and the Total Sums by Partners.

- The first bloc of lines calculates the total of the variables, and generates a new variable (e.g. total_n_employees) in the dataset.
- The second bloc of lines groups by iso_partner and calculates the total sums by partners. (e.g. how many employees are reported by the CBCR countries, say, in the USA)
- The third bloc of lines merges the total sums by partners to the dataset

In [9]:
# First bloc of lines
# Calculate the total sum of profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
total_profit_loss_before_income_tax_corrected = misalignment_2016['profit_loss_before_income_tax_corrected'].sum()
total_n_employees = misalignment_2016['n_employees'].sum()
total_unrelated_party_revenues = misalignment_2016['unrelated_party_revenues'].sum()
total_tangible_assets_except_cash = misalignment_2016['tangible_assets_except_cash'].sum()
total_payroll = misalignment_2016['payroll'].sum()
total_stated_capital = misalignment_2016['stated_capital'].sum()
total_total_revenues = misalignment_2016['total_revenues'].sum()
total_related_party_revenues = misalignment_2016['related_party_revenues'].sum()
total_holding_or_managing_ip = misalignment_2016['holding_or_managing_ip'].sum()

misalignment_2016['total_profit_loss_before_income_tax_corrected'] = total_profit_loss_before_income_tax_corrected
misalignment_2016['total_n_employees'] = total_n_employees
misalignment_2016['total_unrelated_party_revenues'] = total_unrelated_party_revenues
misalignment_2016['total_tangible_assets_except_cash'] = total_tangible_assets_except_cash
misalignment_2016['total_payroll'] = total_payroll
misalignment_2016['total_stated_capital'] = total_stated_capital
misalignment_2016['total_total_revenues'] = total_total_revenues
misalignment_2016['total_related_party_revenues'] = total_related_party_revenues
misalignment_2016['total_holding_or_managing_ip'] = total_holding_or_managing_ip

# Second bloc of lines
# Group by iso_partner and calculate the total sum of profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
total_profit_loss_by_partner = misalignment_2016.groupby('iso_partner')['profit_loss_before_income_tax_corrected'].sum().reset_index()
total_profit_loss_by_partner = total_profit_loss_by_partner.rename(columns={'profit_loss_before_income_tax_corrected': 'total_profit_loss_by_partner'})

total_n_employees_by_partner = misalignment_2016.groupby('iso_partner')['n_employees'].sum().reset_index()
total_n_employees_by_partner = total_n_employees_by_partner.rename(columns={'n_employees': 'total_n_employees_by_partner'})

total_unrelated_party_revenues_by_partner = misalignment_2016.groupby('iso_partner')['unrelated_party_revenues'].sum().reset_index()
total_unrelated_party_revenues_by_partner = total_unrelated_party_revenues_by_partner.rename(columns={'unrelated_party_revenues': 'total_unrelated_party_revenues_by_partner'})

total_tangible_assets_except_cash_by_partner = misalignment_2016.groupby('iso_partner')['tangible_assets_except_cash'].sum().reset_index()
total_tangible_assets_except_cash_by_partner = total_tangible_assets_except_cash_by_partner.rename(columns={'tangible_assets_except_cash': 'total_tangible_assets_except_cash_by_partner'})

total_payroll_by_partner = misalignment_2016.groupby('iso_partner')['payroll'].sum().reset_index()
total_payroll_by_partner = total_payroll_by_partner.rename(columns={'payroll': 'total_payroll_by_partner'})

total_stated_capital_by_partner = misalignment_2016.groupby('iso_partner')['stated_capital'].sum().reset_index()
total_stated_capital_by_partner = total_stated_capital_by_partner.rename(columns={'stated_capital': 'total_stated_capital_by_partner'})

total_total_revenues_by_partner = misalignment_2016.groupby('iso_partner')['total_revenues'].sum().reset_index()
total_total_revenues_by_partner = total_total_revenues_by_partner.rename(columns={'total_revenues': 'total_total_revenues_by_partner'})

total_related_party_revenues_by_partner = misalignment_2016.groupby('iso_partner')['related_party_revenues'].sum().reset_index()
total_related_party_revenues_by_partner = total_related_party_revenues_by_partner.rename(columns={'related_party_revenues': 'total_related_party_revenues_by_partner'})

total_holding_or_managing_ip_by_partner = misalignment_2016.groupby('iso_partner')['holding_or_managing_ip'].sum().reset_index()
total_holding_or_managing_ip_by_partner = total_holding_or_managing_ip_by_partner.rename(columns={'holding_or_managing_ip': 'total_holding_or_managing_ip_by_partner'})

# Merge the totals by partner back into the misalignment_2016 dataframe
misalignment_2016 = misalignment_2016.merge(total_profit_loss_by_partner, on='iso_partner', how='left')
misalignment_2016 = misalignment_2016.merge(total_n_employees_by_partner, on='iso_partner', how='left')
misalignment_2016 = misalignment_2016.merge(total_unrelated_party_revenues_by_partner, on='iso_partner', how='left')
misalignment_2016 = misalignment_2016.merge(total_tangible_assets_except_cash_by_partner, on='iso_partner', how='left')
misalignment_2016 = misalignment_2016.merge(total_payroll_by_partner, on='iso_partner', how='left')
misalignment_2016 = misalignment_2016.merge(total_stated_capital_by_partner, on='iso_partner', how='left')
misalignment_2016 = misalignment_2016.merge(total_total_revenues_by_partner, on='iso_partner', how='left')
misalignment_2016 = misalignment_2016.merge(total_related_party_revenues_by_partner, on='iso_partner', how='left')
misalignment_2016 = misalignment_2016.merge(total_holding_or_managing_ip_by_partner, on='iso_partner', how='left')

misalignment_2016[misalignment_2016['iso_partner'] == 'USA']

,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,total_profit_loss_before_income_tax_corrected,total_n_employees,total_unrelated_party_revenues,total_tangible_assets_except_cash,total_payroll,total_stated_capital,total_total_revenues,total_related_party_revenues,total_holding_or_managing_ip,total_profit_loss_by_partner,total_n_employees_by_partner,total_unrelated_party_revenues_by_partner,total_tangible_assets_except_cash_by_partner,total_payroll_by_partner,total_stated_capital_by_partner,total_total_revenues_by_partner,total_related_party_revenues_by_partner,total_holding_or_managing_ip_by_partner
53,AUS,USA,2016,"-5,424,987,513","3,433,734,906","-2,611,916,814","80,042","38,046,185,219","46,113,091,018","3,683,149,599","154,358,000,000","45,889,856,265","7,843,671,044",14,"2,234,653,761,818","78,780,594","27,782,051,560,972","21,358,997,508,086","1,978,432,767,300","37,074,353,840,708","39,284,072,704,211","11,502,147,534,610","4,230","307,998,782,454","18,845,856","9,340,430,753,791","5,095,086,910,196","867,196,059,161","13,425,902,277,042","12,525,719,027,196","3,185,369,515,182",462
94,BEL,USA,2016,"-5,807,245,489","17,055,562,825","11,241,259,433","116,500","60,319,781,659","20,904,495,382","5,360,772,198","100,507,000,000","74,866,480,183","14,546,809,185",13,"2,234,653,761,818","78,780,594","27,782,051,560,972","21,358,997,508,086","1,978,432,767,300","37,074,353,840,708","39,284,072,704,211","11,502,147,534,610","4,230","307,998,782,454","18,845,856","9,340,430,753,791","5,095,086,910,196","867,196,059,161","13,425,902,277,042","12,525,719,027,196","3,185,369,515,182",462
163,BMU,USA,2016,"-8,495,629,963","20,459,829,797","1,480,245,054","71,989","50,180,677,720","15,512,476,989","3,312,589,097","33,681,340,176","60,570,633,544","10,389,955,823",16,"2,234,653,761,818","78,780,594","27,782,051,560,972","21,358,997,508,086","1,978,432,767,300","37,074,353,840,708","39,284,072,704,211","11,502,147,534,610","4,230","307,998,782,454","18,845,856","9,340,430,753,791","5,095,086,910,196","867,196,059,161","13,425,902,277,042","12,525,719,027,196","3,185,369,515,182",462
204,BRA,USA,2016,"-131,783,852","2,152,134,918","725,127,245","78,336","38,862,562,248","19,868,018,473","3,604,647,647","20,002,953,197","53,832,993,115","14,970,430,867",18,"2,234,653,761,818","78,780,594","27,782,051,560,972","21,358,997,508,086","1,978,432,767,300","37,074,353,840,708","39,284,072,704,211","11,502,147,534,610","4,230","307,998,782,454","18,845,856","9,340,430,753,791","5,095,086,910,196","867,196,059,161","13,425,902,277,042","12,525,719,027,196","3,185,369,515,182",462
217,CAN,USA,2016,"-5,877,150,752","23,459,201,220","16,072,378,000","436,390","228,876,000,000","343,952,000,000","20,080,578,365","623,652,000,000","287,541,000,000","58,665,121,000",NaN,"2,234,653,761,818","78,780,594","27,782,051,560,972","21,358,997,508,086","1,978,432,767,300","37,074,353,840,708","39,284,072,704,211","11,502,147,534,610","4,230","307,998,782,454","18,845,856","9,340,430,753,791","5,095,086,910,196","867,196,059,161","13,425,902,277,042","12,525,719,027,196","3,185,369,515,182",462
231,CHL,USA,2016,0,"202,989,684","208,708,220","5,985","3,692,185,585","1,313,330,648","275,401,044","2,527,248,868","4,044,031,401","351,845,816",1,"2,234,653,761,818","78,780,594","27,782,051,560,972","21,358,997,508,086","1,978,432,767,300","37,074,353,840,708","39,284,072,704,211","11,502,147,534,610","4,230","307,998,782,454","18,845,856","9,340,430,753,791","5,095,086,910,196","867,196,059,161","13,425,902,277,042","12,525,719,027,196","3,185,369,515,182",462
315,CHN,USA,2016,"-3,165,530,873","2,359,320,751","-1,046,732,590","19,867","29,447,116,804","26,568,223,047","914,184,217","40,038,151,211","35,970,331,218","6,523,214,414",6,"2,234,653,761,818","78,780,594",

### Step 4.2. Calculate the shares for all the variables

In [10]:
# Final Misalignment
final_misalignment_2016 = misalignment_2016

# Calculate the shares for all variables
final_misalignment_2016['share_reported_total_profit_loss_by_partner'] = misalignment_2016['total_profit_loss_by_partner'] / misalignment_2016['total_profit_loss_before_income_tax_corrected']
final_misalignment_2016['share_reported_total_n_employees_by_partner'] = misalignment_2016['total_n_employees_by_partner'] / misalignment_2016['total_n_employees']
final_misalignment_2016['share_reported_total_unrelated_party_revenues_by_partner'] = misalignment_2016['total_unrelated_party_revenues_by_partner'] / misalignment_2016['total_unrelated_party_revenues']
final_misalignment_2016['share_reported_total_tangible_assets_except_cash_by_partner'] = misalignment_2016['total_tangible_assets_except_cash_by_partner'] / misalignment_2016['total_tangible_assets_except_cash']
final_misalignment_2016['share_reported_total_payroll_by_partner'] = misalignment_2016['total_payroll_by_partner'] / misalignment_2016['total_payroll']
final_misalignment_2016['share_reported_total_stated_capital_by_partner'] = misalignment_2016['total_stated_capital_by_partner'] / misalignment_2016['total_stated_capital']
final_misalignment_2016['share_reported_total_total_revenues_by_partner'] = misalignment_2016['total_total_revenues_by_partner'] / misalignment_2016['total_total_revenues']
final_misalignment_2016['share_reported_total_related_party_revenues_by_partner'] = misalignment_2016['total_related_party_revenues_by_partner'] / misalignment_2016['total_related_party_revenues']
final_misalignment_2016['share_reported_total_holding_or_managing_ip_by_partner'] = misalignment_2016['total_holding_or_managing_ip_by_partner'] / misalignment_2016['total_holding_or_managing_ip']

# Give me column 'share reported' with 2 decimals
pd.options.display.float_format = '{:,.2f}'.format

final_misalignment_2016[final_misalignment_2016['iso_partner'] == 'USA']

,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,total_profit_loss_before_income_tax_corrected,total_n_employees,total_unrelated_party_revenues,total_tangible_assets_except_cash,total_payroll,total_stated_capital,total_total_revenues,total_related_party_revenues,total_holding_or_managing_ip,total_profit_loss_by_partner,total_n_employees_by_partner,total_unrelated_party_revenues_by_partner,total_tangible_assets_except_cash_by_partner,total_payroll_by_partner,total_stated_capital_by_partner,total_total_revenues_by_partner,total_related_party_revenues_by_partner,total_holding_or_managing_ip_by_partner,share_reported_total_profit_loss_by_partner,share_reported_total_n_employees_by_partner,share_reported_total_unrelated_party_revenues_by_partner,share_reported_total_tangible_assets_except_cash_by_partner,share_reported_total_payroll_by_partner,share_reported_total_stated_capital_by_partner,share_reported_total_total_revenues_by_partner,share_reported_total_related_party_revenues_by_partner,share_reported_total_holding_or_managing_ip_by_partner
53,AUS,USA,2016,"-5,424,987,513.06","3,433,734,906.14","-2,611,916,814.00","80,042.00","38,046,185,219.00","46,113,091,018.00","3,683,149,598.90","154,358,000,000.00","45,889,856,265.00","7,843,671,044.00",14.00,"2,234,653,761,817.55","78,780,594.24","27,782,051,560,972.21","21,358,997,508,086.03","1,978,432,767,299.78","37,074,353,840,708.30","39,284,072,704,210.77","11,502,147,534,610.24","4,230.00","307,998,782,453.94","18,845,856.00","9,340,430,753,790.98","5,095,086,910,195.73","867,196,059,161.47","13,425,902,277,041.68","12,525,719,027,195.84","3,185,369,515,181.56",462.00,0.14,0.24,0.34,0.24,0.44,0.36,0.32,0.28,0.11
94,BEL,USA,2016,"-5,807,245,488.89","17,055,562,824.76","11,241,259,433.00","116,500.00","60,319,781,659.00","20,904,495,382.00","5,360,772,198.00","100,507,000,000.00","74,866,480,183.00","14,546,809,185.00",13.00,"2,234,653,761,817.55","78,780,594.24","27,782,051,560,972.21","21,358,997,508,086.03","1,978,432,767,299.78","37,074,353,840,708.30","39,284,072,704,210.77","11,502,147,534,610.24","4,230.00","307,998,782,453.94","18,845,856.00","9,340,430,753,790.98","5,095,086,910,195.73","867,196,059,161.47","13,425,902,277,041.68","12,525,719,027,195.84","3,185,369,515,181.56",462.00,0.14,0.24,0.34,0.24,0.44,0.36,0.32,0.28,0.11
163,BMU,USA,2016,"-8,495,629,963.18","20,459,829,796.55","1,480,245,054.00","71,989.00","50,180,677,720.00","15,512,476,989.00","3,312,589,096.67","33,681,340,176.00","60,570,633,544.00","10,389,955,823.00",16.00,"2,234,653,761,817.55","78,780,594.24","27,782,051,560,972.21","21,358,997,508,086.03","1,978,432,767,299.78","37,074,353,840,708.30","39,284,072,704,210.77","11,502,147,534,610.24","4,230.00","307,998,782,453.94","18,845,856.00","9,340,430,753,790.98","5,095,086,910,195.73","867,196,059,161.47","13,425,902,277,041.68","12,525,719,027,195.84","3,185,369,515,181.56",462.00,0.14,0.24,0.34,0.24,0.44,0.36,0.32,0.28,0.11
204,BRA,USA,2016,"-131,783,851.66","2,152,134,917.83","725,127,245.00","78,336.00","38,862,562,248.00","19,868,018,473.00","3,604,647,647.23","20,002,953,197.00","53,832,993,115.00","14,970,430,867.00",18.00,"2,234,653,761,817.55","78,780,594.24","27,782,051,560,972.21","21,358,997,508,086.03","1,978,432,767,299.78","37,074,353,840,708.30","39,284,072,704,210.77","11,502,147,534,610.24","4,230.00","307,998,782,453.94","18,845,856.00","9,340,430,753,790.98","5,095,086,910,195.73","867,196,059,161.47","13,425,902,277,041.68","12,525,719,027,195.84","3,185,369,515,181.56",462.00,0.14,0.24,0.34,0.24,0.44,0.36,0.32,0.28,0.11
217,CAN,USA,2016,"-5,877,150,751.98","23,459,201,220.20","16,072,378,000.00","436,390.00","228,876,000,000.00","343,952,000,000.00","20,080,578,364.68","623,652,000,000.00","287,541,000,000.00","58,665,121,000.0

### Step 4.3. Keep only the shares for the partners, and drop duplicates (effectively only keep one value for each iso_partner)

In [11]:
# Keep iso_partner and share_reported_total_profit_loss_by_partner	share_reported_total_n_employees_by_partner	share_reported_total_unrelated_party_revenues_by_partner	share_reported_total_tangible_assets_except_cash_by_partner	share_reported_total_payroll_by_partner	share_reported_total_stated_capital_by_partner	share_reported_total_total_revenues_by_partner	share_reported_total_related_party_revenues_by_partner	share_reported_total_holding_or_managing_ip_by_partner
shares_reported_2016 = final_misalignment_2016[['iso_partner', 'share_reported_total_profit_loss_by_partner', 'share_reported_total_n_employees_by_partner', 'share_reported_total_unrelated_party_revenues_by_partner', 'share_reported_total_tangible_assets_except_cash_by_partner', 'share_reported_total_payroll_by_partner', 'share_reported_total_stated_capital_by_partner', 'share_reported_total_total_revenues_by_partner', 'share_reported_total_related_party_revenues_by_partner', 'share_reported_total_holding_or_managing_ip_by_partner']]

# Drop duplicates
shares_reported_2016 = shares_reported_2016.drop_duplicates()

shares_reported_2016[shares_reported_2016['iso_partner'] == 'ESP']

,iso_partner,share_reported_total_profit_loss_by_partner,share_reported_total_n_employees_by_partner,share_reported_total_unrelated_party_revenues_by_partner,share_reported_total_tangible_assets_except_cash_by_partner,share_reported_total_payroll_by_partner,share_reported_total_stated_capital_by_partner,share_reported_total_total_revenues_by_partner,share_reported_total_related_party_revenues_by_partner,share_reported_total_holding_or_managing_ip_by_partner
17,ESP,0.01,0.01,0.01,0.00,0.01,0.00,0.01,0.01,0.01


### Step 4.4. Bring back the excluded countries

- The key here is the third command, where we sum by iso_parent. We basically assume that all countries report for the rest of the world, without caring about continents. **This could be improved and changed**. 
- Once that sum is done, we combine all potential iso_combinations for 2016 (created in Step 1), and we keep all combinations for the "excluded countries".
- Note in the test view, that no matter who is the iso_partner, the number in the variables will be the same because we have aggregated them. The next cells will now create the right shares.

In [12]:
# 1.Call dataset
excluded_2016 = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# 2.Keep if year == 2016 and iso_parent == 'AUT', 'FIN', 'IRL0', 'KOR', 'NLD', 'NOR' 'SWE'
excluded_2016 = excluded_2016[excluded_2016['year'] == 2016]
excluded_2016 = excluded_2016[excluded_2016['iso_parent'].isin(['AUT', 'FIN', 'IRL', 'KOR', 'NLD', 'NOR', 'SWE'])]

# 3. Sum by iso_parent: 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip' and 'profit_loss_before_income_tax_corrected'
excluded_2016 = excluded_2016.groupby('iso_parent').agg({'n_employees': 'sum', 'unrelated_party_revenues': 'sum', 'tangible_assets_except_cash': 'sum', 'payroll': 'sum', 'stated_capital': 'sum', 'total_revenues': 'sum', 'related_party_revenues': 'sum', 'holding_or_managing_ip': 'sum', 'profit_loss_before_income_tax_corrected': 'sum'}).reset_index()

# 4. Merge iso_combinations_2016 with excluded_2016. 
excluded_jurisdictions_2016 = pd.merge(iso_combinations_2016, excluded_2016, on='iso_parent', how='left')

# Drop year_y
excluded_jurisdictions_2016 = excluded_jurisdictions_2016.drop(columns=['year_y'])
# Rename year_x to year
excluded_jurisdictions_2016 = excluded_jurisdictions_2016.rename(columns={'year_x': 'year'})

# 5. Keep if iso_parent == 'AUT', 'FIN', 'IRL0', 'KOR', 'NLD', 'NOR' 'SWE'
excluded_jurisdictions_2016 = excluded_jurisdictions_2016[excluded_jurisdictions_2016['iso_parent'].isin(['AUT', 'FIN', 'IRL', 'KOR', 'NLD', 'NOR', 'SWE'])]

# 6. Test view.
excluded_jurisdictions_2016[excluded_jurisdictions_2016['iso_partner'] == 'ESP']


,iso_parent,year,iso_partner,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,profit_loss_before_income_tax_corrected
232,AUT,2016,ESP,"1,642,504.00","595,515,485,297.70","321,669,523,402.00","11,220,274,767.65","251,168,133,983.80","773,401,121,859.00","177,917,146,078.80",451.00,"40,892,944,644.90"
1696,FIN,2016,ESP,"574,440.00","212,203,720,328.00","85,594,738,489.00","8,679,716,706.98","272,663,996,317.00","286,935,000,000.00","74,736,508,828.00",161.00,"25,179,380,440.42"
2245,IRL,2016,ESP,"675,541.00","243,658,701,311.00","132,780,656,599.00","3,460,389,921.46","1,979,041,000,000.00","391,436,009,406.00","168,692,880,218.00",445.00,"40,888,035,863.81"
2794,KOR,2016,ESP,"2,813,456.00","1,516,352,000,000.00","1,003,830,000,000.00","50,888,795,675.74","333,624,000,000.00","2,179,797,000,000.00","670,960,000,000.00",452.00,"104,132,507,017.50"
3343,NLD,2016,ESP,"3,672,779.00","1,387,430,000,000.00","812,177,000,000.00","29,960,639,368.80","2,186,659,000,000.00","2,208,954,000,000.00","823,648,000,000.00",925.00,"80,833,124,737.93"
3526,NOR,2016,ESP,"672,690.00","342,765,588,000.00","366,517,883,000.00","12,025,313,493.84","706,424,401,000.00","462,498,166,000.00","118,626,206,000.00",205.00,"37,061,095,205.00"
4258,SWE,2016,ESP,"3,213,592.00","1,033,521,226,758.00","572,745,042,454.00","16,525,422,222.62","577,420,358,478.00","1,547,807,469,018.00","514,285,642,155.00","1,166.00","98,068,647,975.01"


### Step 4.5. Merge with the shares reported, and multiply the number

In [13]:
# Merge with share_reported_2016
excluded_jurisdictions_share_reported_2016 = pd.merge(excluded_jurisdictions_2016, shares_reported_2016, on='iso_partner', how='left')

# Multiply 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip' and 'profit_loss_before_income_tax_corrected' by share_reported
excluded_jurisdictions_share_reported_2016['n_employees'] = excluded_jurisdictions_share_reported_2016['n_employees'] * excluded_jurisdictions_share_reported_2016['share_reported_total_n_employees_by_partner']
excluded_jurisdictions_share_reported_2016['unrelated_party_revenues'] = excluded_jurisdictions_share_reported_2016['unrelated_party_revenues'] * excluded_jurisdictions_share_reported_2016['share_reported_total_unrelated_party_revenues_by_partner']
excluded_jurisdictions_share_reported_2016['tangible_assets_except_cash'] = excluded_jurisdictions_share_reported_2016['tangible_assets_except_cash'] * excluded_jurisdictions_share_reported_2016['share_reported_total_tangible_assets_except_cash_by_partner']
excluded_jurisdictions_share_reported_2016['payroll'] = excluded_jurisdictions_share_reported_2016['payroll'] * excluded_jurisdictions_share_reported_2016['share_reported_total_payroll_by_partner']
excluded_jurisdictions_share_reported_2016['stated_capital'] = excluded_jurisdictions_share_reported_2016['stated_capital'] * excluded_jurisdictions_share_reported_2016['share_reported_total_stated_capital_by_partner']
excluded_jurisdictions_share_reported_2016['total_revenues'] = excluded_jurisdictions_share_reported_2016['total_revenues'] * excluded_jurisdictions_share_reported_2016['share_reported_total_total_revenues_by_partner']
excluded_jurisdictions_share_reported_2016['related_party_revenues'] = excluded_jurisdictions_share_reported_2016['related_party_revenues'] * excluded_jurisdictions_share_reported_2016['share_reported_total_related_party_revenues_by_partner']
excluded_jurisdictions_share_reported_2016['holding_or_managing_ip'] = excluded_jurisdictions_share_reported_2016['holding_or_managing_ip'] * excluded_jurisdictions_share_reported_2016['share_reported_total_holding_or_managing_ip_by_partner']
excluded_jurisdictions_share_reported_2016['profit_loss_before_income_tax_corrected'] = excluded_jurisdictions_share_reported_2016['profit_loss_before_income_tax_corrected'] * excluded_jurisdictions_share_reported_2016['share_reported_total_profit_loss_by_partner']

# Drop share_reported columns
excluded_jurisdictions_dataset_2016 = excluded_jurisdictions_share_reported_2016.drop(columns=[
    'share_reported_total_profit_loss_by_partner',
    'share_reported_total_n_employees_by_partner',
    'share_reported_total_unrelated_party_revenues_by_partner',
    'share_reported_total_tangible_assets_except_cash_by_partner',
    'share_reported_total_payroll_by_partner',
    'share_reported_total_stated_capital_by_partner',
    'share_reported_total_total_revenues_by_partner',
    'share_reported_total_related_party_revenues_by_partner',
    'share_reported_total_holding_or_managing_ip_by_partner'
])

# Test view.
excluded_jurisdictions_dataset_2016[excluded_jurisdictions_dataset_2016['iso_partner'] == 'ESP']

,iso_parent,year,iso_partner,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,profit_loss_before_income_tax_corrected
49,AUT,2016,ESP,"12,471.83","3,759,066,731.27","1,479,349,224.69","84,631,621.58","889,672,667.38","5,096,382,103.73","1,291,079,144.00",2.67,"264,450,647.66"
232,FIN,2016,ESP,"4,361.83","1,339,491,524.62","393,647,830.49","65,468,851.25","965,814,018.89","1,890,778,740.30","542,335,294.66",0.95,"162,832,574.74"
415,IRL,2016,ESP,"5,129.51","1,538,044,501.77","610,654,560.36","26,100,823.41","7,010,040,077.11","2,579,395,628.88","1,224,142,046.97",2.63,"264,418,903.06"
598,KOR,2016,ESP,"21,363.09","9,571,654,300.88","4,616,586,353.96","383,840,983.19","1,181,742,879.85","14,363,928,505.65","4,868,909,385.93",2.67,"673,414,672.44"
781,NLD,2016,ESP,"27,888.08","8,757,861,187.02","3,735,179,517.65","225,985,329.77","7,745,452,077.53","14,556,060,646.14","5,976,909,916.98",5.47,"522,739,860.75"
964,NOR,2016,ESP,"5,107.86","2,163,635,959.57","1,685,605,587.74","90,703,819.84","2,502,254,052.57","3,047,664,801.09","860,826,648.10",1.21,"239,670,454.53"
1147,SWE,2016,ESP,"24,401.39","6,523,886,205.28","2,634,038,579.53","124,646,972.47","2,045,303,687.12","10,199,388,211.60","3,731,981,325.47",6.89,"634,200,292.92"


## Step 5. Calculate Misalignment with all countries reporting in the CBCR

### Step 5.1. Concatenate the two samples

- Concatenate the cbcr_sample (without the "bad reporters") and the sample with the bad reporters.
- Ensure all required columns are included: 'iso_parent', 'year', 'iso_partner', 'n_employees', 'unrelated_party_revenues', 'tangible_assets_except_cash', 'payroll', 'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip', 'profit_loss_before_income_tax_corrected'.

In [14]:
final_misalignment_2016 = cbcr_sample[cbcr_sample['year'] == 2016].copy()

# Concatenate excluded_jurisdictions_dataset_2016
final_misalignment_2016 = pd.concat([final_misalignment_2016, excluded_jurisdictions_dataset_2016])

# Save the final misalignment dataset
final_misalignment_2016.to_csv(f'{output_tables}/Datasets_to_Perform_Analysis/misalignment_dataset_2016.csv', index=False)

final_misalignment_2016[final_misalignment_2016['iso_partner'] == 'ESP']

,iso_parent,parent_jurisdiction,iso_partner,partner_jurisdiction,year,unrelated_party_revenues,profit_loss_before_income_tax,adjusted_profit_loss_before_income_tax,income_tax_paid_on_cash_basis,income_tax_accrued_current_year,n_employees,tangible_assets_except_cash,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,n_cbcr,n_cbcr_groups,n_entities,profit_loss_before_income_tax_corrected,ln_profit_loss_before_income_tax_corrected,ln_unrelated_party_revenues,ln_n_employees,ln_tangible_assets_except_cash,ln_stated_capital,ln_total_revenues,ln_related_party_revenues,ln_holding_or_managing_ip,etr_domestic,etr_domestic_corrected,etr_foreign,etr_foreign_corrected,etr_average,etr_average_corrected,cit,gdp_current_usd,population,gdp,wage_monthly,payroll,ln_wage_monthly,ln_gdp_current_usd,ln_population,gvt_health_expenditure,ln_gvt_health_expenditure,tax_revenue_pct_gdp,tax_revenue_current_usd,cthi_2021_share,cthi_2021_score,region_tjn,ukt,gbr_oct,nld_oct,oecd_oct,oecd,eu
395,AUS,Australia,ESP,Spain,2016,"335,589,593.00","24,908,978.66",NaN,"5,727,348.25","1,433,598.19","1,496.00","186,498,778.30","265,166,903.20","367,421,719.50","31,832,125.78",NaN,16.00,16.00,30.00,"24,908,978.66",17.03,19.63,7.31,19.04,19.40,19.72,17.28,NaN,0.09,0.09,0.13,0.13,0.11,0.11,0.25,"1,233,554,967,011.68","46,484,062.00",NaN,"2,078.86","37,319,766.53",7.64,27.84,17.65,"78,972,636,876.33",25.09,13.79,"170,092,266,805.81",0.02,64.81,Europe,0.00,0.00,0.00,0.00,1.00,1.00
884,BEL,Belgium,ESP,Spain,2016,"1,731,399,398.00","842,239,602.50",NaN,"92,955,100.00","54,666,451.67","5,500.00","1,263,968,038.00","1,472,010,405.00","2,600,418,923.00","869,019,524.20",NaN,27.00,27.00,76.00,"842,239,602.50",20.55,21.27,8.61,20.96,21.11,21.68,20.58,NaN,0.09,0.09,0.13,0.13,0.11,0.11,0.25,"1,233,554,967,011.68","46,484,062.00",NaN,"2,078.86","137,205,024.00",7.64,27.84,17.65,"78,972,636,876.33",25.09,13.79,"170,092,266,805.81",0.02,64.81,Europe,0.00,0.00,0.00,0.00,1.00,1.00
1234,BMU,Bermuda,ESP,Spain,2016,"2,241,442,601.00","66,235,371.00",NaN,"20,623,392.00","-38,276,189.00",994.00,"409,811,406.00","114,518,750.00","4,357,717,448.00","2,116,274,848.00",NaN,18.00,18.00,35.00,"66,235,371.00",18.01,21.53,6.90,19.83,18.56,22.20,21.47,NaN,0.09,0.09,0.13,0.13,0.11,0.11,0.25,"1,233,554,967,011.68","46,484,062.00",NaN,"2,078.86","24,796,689.79",7.64,27.84,17.65,"78,972,636,876.33",25.09,13.79,"170,092,266,805.81",0.02,64.81,Europe,0.00,0.00,0.00,0.00,1.00,1.00
1745,BRA,Brazil,ESP,Spain,2016,"699,198,341.00","-995,227,152.00",NaN,"17,561,325.00","-899,838.00","1,589.00","520,065,247.00","2,093,243,875.00","1,078,697,298.00","379,498,957.00",0.00,17.00,17.00,51.00,"-995,227,152.00",0.00,20.37,7.37,20.07,21.46,20.80,19.75,0.00,0.09,0.09,0.13,0.13,0.11,0.11,0.25,"1,233,554,967,011.68","46,484,062.00",NaN,"2,078.86","39,639,778.75",7.64,27.84,17.65,"78,972,636,876.33",25.09,13.79,"170,092,266,805.81",0.02,64.81,Europe,0.00,0.00,0.00,0.00,1.00,1.00
1917,CAN,Canada,ESP,Spain,2016,NaN,"244,568,000.00",NaN,"44,708,000.00","65,927,000.00","6,040.00","1,806,482,000.00","3,867,383,000.00",NaN,NaN,NaN,40.00,40.00,90.00,"244,568,000.00",19.32,NaN,8.71,21.31,22.08,NaN,NaN,NaN,0.09,0.09,0.13,0.13,0.11,0.11,0.25,"1,233,554,967,011.68","46,484,062.00",NaN,"2,078.86","150,676,062.72",7.64,27.84,17.65,"78,972,636,876.33",25.09,13.79,"170,092,266,805.81",0.02,64.81,Europe,0.00,0.00,0.00,0.00,1.00,1.00
3018,CHN,China (People’s Republic of),ESP,Spain,2016,"1,667,424,443.00","32,616,302.86",NaN,"13,974,285.22","11,969,779.32","1,901.00","259,452,928.70","181,924,612.00","1,774,094,143.00","106,669,699.90",0.00,13.00,13.00,20.00,"32,616,302.86",17.30,21.23,7.55,19.37,19.02,21.30,18.49,0.00,0.09,0.09,0.13,0.13,0.11,0.11,0.25,"1,233,554,967,011.68","46,484,062.00",NaN,"2,078.86","47,423,045.57",7.64,27.84,17.65,"78,972,636,876.33",25.09,13.79,"170,092,266,805.81",0.02,64.81,Europe,0.00,0.00,0.00,0.00,1.00,1.00
5134,DNK,Denmark,ESP,Spain,2016,"1,963,081,394.00","-52,835,722.47",NaN,"-27,8

### Step 5.2. Perform the misalignment estimates, and all the remaining calculations needed

In [15]:
# 1. Initialize a list to store the aggregate results
results_sample = []

# 2. Run the estimates
misalignment_final_estimates_2016 = final_misalignment_2016[final_misalignment_2016['year'] == 2016].copy()
misalignment_final_estimates_2016 = calculate_misalignment(misalignment_final_estimates_2016, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# 3. Perform the groupby operation on 'iso_partner'
country_results_2016 = misalignment_final_estimates_2016.groupby(['iso_partner']).agg(
    negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
    positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
    theoretical_profit=('theoretical_profit', 'sum'),
    reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
).reset_index()

# 4. Convert results to millions
country_results_2016['negative_misalignment'] = -country_results_2016['negative_misalignment'] / 1e6
country_results_2016['positive_misalignment'] = country_results_2016['positive_misalignment'] / 1e6
country_results_2016['theoretical_profit'] = country_results_2016['theoretical_profit'] / 1e6
country_results_2016['reported_profit'] = country_results_2016['reported_profit'] / 1e6

# 5. Merge the unique columns back into the result
country_results_2016 = country_results_2016.merge(unique_columns, on='iso_partner', how='left')

# 6. Calculate other relevant variables
country_results_2016['tax_revenue_loss'] = country_results_2016['negative_misalignment'] * country_results_2016['cit']
country_results_2016['tax_revenue_gain'] = country_results_2016['positive_misalignment'] * country_results_2016['etr_average_corrected']

country_results_2016['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
    country_results_2016['gvt_health_expenditure'] == 0, 
    np.nan, 
    country_results_2016['tax_revenue_loss'] / (country_results_2016['gvt_health_expenditure'] / 1e6)
)
    
country_results_2016['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
    country_results_2016['tax_revenue_current_usd'] == 0, 
    np.nan, 
    country_results_2016['tax_revenue_loss'] / (country_results_2016['tax_revenue_current_usd'] / 1e6)
)

# 7. Calculate totals
total_positive_misalignment = country_results_2016['positive_misalignment'].sum()
total_negative_misalignment = country_results_2016['negative_misalignment'].sum()
total_profits = country_results_2016['reported_profit'].sum()
misaligned_of_total_profits = total_positive_misalignment / total_profits
total_tax_revenue_loss = country_results_2016['tax_revenue_loss'].sum()
total_tax_revenue_gain = country_results_2016['tax_revenue_gain'].sum()
average_tax_revenue_loss_pct_of_gvt_health_expenditure = country_results_2016['tax_revenue_loss_pct_of_gvt_health_expenditure'].mean()
average_tax_revenue_loss_pct_of_total_tax_revenues = country_results_2016['tax_revenue_loss_pct_of_total_tax_revenues'].mean()


print(f"Year {2016}: Positive Misalignment: {total_positive_misalignment}, Negative Misalignment: {total_negative_misalignment}, Shifted of total profits: {misaligned_of_total_profits}, "
        f"Total tax revenue loss: {total_tax_revenue_loss}, Total tax revenue gain: {total_tax_revenue_gain}")

# 9. Calculate countries' fractions of totals
country_results_2016['tax_revenue_loss_caused_pct_of_total'] = country_results_2016['positive_misalignment'] / total_positive_misalignment
country_results_2016['tax_revenue_loss_caused_usd'] = country_results_2016['tax_revenue_loss_caused_pct_of_total'] * total_tax_revenue_loss
country_results_2016['tax_revenue_loss_suffered_pct_of_total'] = country_results_2016['tax_revenue_loss'] / total_tax_revenue_loss

#country_results_2016 = country_results_2016[['iso_partner', 'partner_jurisdiction', 'negative_misalignment',
#   'tax_revenue_loss', 'tax_revenue_loss_suffered_pct_of_total', 'tax_revenue_loss_pct_of_gvt_health_expenditure',
#   'tax_revenue_loss_pct_of_total_tax_revenues', 'positive_misalignment', 'tax_revenue_gain', 
#   'tax_revenue_loss_caused_usd', 'tax_revenue_loss_caused_pct_of_total', 'etr_average_corrected', 'cit',
#   'region_tjn', 'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

country_results_2016 = country_results_2016.sort_values(by='iso_partner')
country_results_2016.to_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_countries_2016.csv', index=False) # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE
    
# 10. Append aggregate results to the list
results_sample.append({
    'year': 2016,
    'total_positive_misalignment': total_positive_misalignment,
    'total_negative_misalignment': total_negative_misalignment,
    'total_profits': total_profits,
    'misaligned_of_total_profits': misaligned_of_total_profits,
    'total_tax_revenue_loss': total_tax_revenue_loss,
    'total_tax_revenue_gain': total_tax_revenue_gain,
    'average_tax_revenue_loss_pct_of_gvt_health_expenditure': average_tax_revenue_loss_pct_of_gvt_health_expenditure,
    'average_tax_revenue_loss_pct_of_total_tax_revenues': average_tax_revenue_loss_pct_of_total_tax_revenues
})

# 11. Convert aggregate results to a DataFrame
results_sample_df = pd.DataFrame(results_sample)

# 12. Save the aggregated results to a CSV or Excel file
results_sample_df.to_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_aggregate_results_2016.csv', index=False)  # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE

Year 2016: Positive Misalignment: 530615.3114510451, Negative Misalignment: 530615.3114510451, Shifted of total profits: 0.19935132361706898, Total tax revenue loss: 147413.25389962248, Total tax revenue gain: 40946.056384616044


C:\Users\aliso\AppData\Local\Temp\ipykernel_18052\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


In [16]:
# Ensure the inputs are numeric (optional but robust)
cols_num = ['misaligned_profit', 'profit_loss_before_income_tax_corrected']
misalignment_final_estimates_2016[cols_num] = misalignment_final_estimates_2016[cols_num].apply(
    pd.to_numeric, errors='coerce'
).fillna(0)

# By headquarter (reporting) country
hq_all = (
    misalignment_final_estimates_2016
    .groupby('iso_parent', as_index=False)
    .agg(
        shifted_out=('misaligned_profit', lambda s: (-s.clip(upper=0)).sum()),  # magnitude of negatives
        reported_profit=('profit_loss_before_income_tax_corrected', 'sum'),
        shifted_in=('misaligned_profit', lambda s: s.clip(lower=0).sum())       # optional
    )
)

# Fractions (per HQ)
hq_all['fraction_shifted_out'] = np.where(
    hq_all['reported_profit'] == 0, np.nan, hq_all['shifted_out'] / hq_all['reported_profit']
)
hq_all['fraction_shifted_out_pct'] = 100 * hq_all['fraction_shifted_out']

# (optional)
hq_all['fraction_shifted_in'] = np.where(
    hq_all['reported_profit'] == 0, np.nan, hq_all['shifted_in'] / hq_all['reported_profit']
)
hq_all['fraction_shifted_in_pct'] = 100 * hq_all['fraction_shifted_in']

# Save
hq_all.sort_values('iso_parent').to_csv(
    f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_hq_fractions_2016_all.csv', index=False
)

DO everything for US only

In [17]:
import os
import numpy as np
import pandas as pd

# ================== Config (pick ONE year) ==================
YEAR = 2016  # <— change this per file (e.g., 2018, 2019, …)
OUTPUT_DIR = f"{output_tables}/Final_Full_CBCR_Datasets"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ETR_MAX = 0.15
WEIGHTS = [0.5, 0, 0, 0.5, 0, 0, 0, 0]  # same as your example

# ================== Load the per-year df by name ==================
df_name = f"final_misalignment_{YEAR}"
if df_name not in globals():
    raise NameError(f"{df_name} not found in globals(). Make sure you created it earlier.")
final_misalignment_year = globals()[df_name]

# ================== Collect HQs (keeps your year filter) ==================
hq_list = (
    final_misalignment_year.loc[final_misalignment_year["year"] == YEAR, "iso_parent"]
    .dropna().astype(str).str.upper().unique()
)
hq_list = sorted(hq_list)
print(f"[{YEAR}] HQs: {', '.join(hq_list)}")

# ================== Run for this single year ==================
for HQ in hq_list:
    # 1) Initialize a list to store the aggregate results (per HQ & YEAR)
    results_sample = []

    # 2) Run the estimates — EXACT same filter structure
    mask = (final_misalignment_year['year'] == YEAR) & (final_misalignment_year['iso_parent'] == HQ)
    misalignment_final_estimates = final_misalignment_year.loc[mask].copy()

    if misalignment_final_estimates.empty:
        print(f"[{YEAR}][{HQ}] No rows; skipping.")
        continue

    misalignment_final_estimates = calculate_misalignment(
        misalignment_final_estimates,
        etr_max=ETR_MAX,
        weights=WEIGHTS
    )

    # 3) Groupby iso_partner
    country_results = misalignment_final_estimates.groupby(['iso_partner']).agg(
        negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
        positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
        theoretical_profit=('theoretical_profit', 'sum'),
        reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
    ).reset_index()

    # 4) Convert to millions
    country_results['negative_misalignment'] = -country_results['negative_misalignment'] / 1e6
    country_results['positive_misalignment'] =  country_results['positive_misalignment'] / 1e6
    country_results['theoretical_profit']   =  country_results['theoretical_profit'] / 1e6
    country_results['reported_profit']      =  country_results['reported_profit'] / 1e6

    # 5) Merge the unique columns
    country_results = country_results.merge(unique_columns, on='iso_partner', how='left')

    # 6) Other variables (same logic as yours)
    country_results['tax_revenue_loss'] = country_results['negative_misalignment'] * country_results['cit']
    country_results['tax_revenue_gain'] = country_results['positive_misalignment'] * country_results['etr_average_corrected']

    country_results['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
        country_results['gvt_health_expenditure'] == 0, 
        np.nan, 
        country_results['tax_revenue_loss'] / (country_results['gvt_health_expenditure'] / 1e6)
    )
    country_results['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
        country_results['tax_revenue_current_usd'] == 0, 
        np.nan, 
        country_results['tax_revenue_loss'] / (country_results['tax_revenue_current_usd'] / 1e6)
    )

    # 7) Totals
    total_positive_misalignment = country_results['positive_misalignment'].sum()
    total_negative_misalignment = country_results['negative_misalignment'].sum()
    total_profits               = country_results['reported_profit'].sum()
    misaligned_of_total_profits = total_positive_misalignment / total_profits if total_profits != 0 else np.nan
    total_tax_revenue_loss      = country_results['tax_revenue_loss'].sum()
    total_tax_revenue_gain      = country_results['tax_revenue_gain'].sum()
    average_loss_pct_health     = country_results['tax_revenue_loss_pct_of_gvt_health_expenditure'].mean()
    average_loss_pct_taxrev     = country_results['tax_revenue_loss_pct_of_total_tax_revenues'].mean()

    print(
        f"[{YEAR}][{HQ}] +mis={total_positive_misalignment:.3f}m, "
        f"-mis={total_negative_misalignment:.3f}m, "
        f"share={misaligned_of_total_profits if pd.notna(misaligned_of_total_profits) else np.nan:.3f}, "
        f"loss={total_tax_revenue_loss:.3f}m, gain={total_tax_revenue_gain:.3f}m"
    )

    # 9) Fractions of totals
    country_results['tax_revenue_loss_caused_pct_of_total'] = (
        country_results['positive_misalignment'] / total_positive_misalignment
        if total_positive_misalignment != 0 else np.nan
    )
    country_results['tax_revenue_loss_caused_usd'] = (
        country_results['tax_revenue_loss_caused_pct_of_total'] * total_tax_revenue_loss
    )
    country_results['tax_revenue_loss_suffered_pct_of_total'] = (
        country_results['tax_revenue_loss'] / total_tax_revenue_loss
        if total_tax_revenue_loss != 0 else np.nan
    )

    # Save per-HQ country file
    country_results = country_results.sort_values(by='iso_partner')
    per_hq_file = f'{OUTPUT_DIR}/SOTJ_sample_countries_{YEAR}_{HQ}MNEs.csv'
    country_results.to_csv(per_hq_file, index=False)

    # 10) Append aggregate results to the list (per HQ & year)
    results_sample = [{
        'year': YEAR,
        'total_positive_misalignment': total_positive_misalignment,
        'total_negative_misalignment': total_negative_misalignment,
        'total_profits': total_profits,
        'misaligned_of_total_profits': misaligned_of_total_profits,
        'total_tax_revenue_loss': total_tax_revenue_loss,
        'total_tax_revenue_gain': total_tax_revenue_gain,
        'average_tax_revenue_loss_pct_of_gvt_health_expenditure': average_loss_pct_health,
        'average_tax_revenue_loss_pct_of_total_tax_revenues': average_loss_pct_taxrev
    }]

    # 11) Save the aggregated results (per HQ & year)
    results_sample_df = pd.DataFrame(results_sample)
    agg_file = f'{OUTPUT_DIR}/SOTJ_sample_aggregate_results_{YEAR}_{HQ}MNEs.csv'
    results_sample_df.to_csv(agg_file, index=False)


[2016] HQs: AUS, AUT, BEL, BMU, BRA, CAN, CHL, CHN, DNK, FIN, FRA, IDN, IRL, ITA, JPN, KOR, LUX, MEX, NLD, NOR, POL, SGP, SVN, SWE, USA, ZAF


[2016][AUS] +mis=10992.857m, -mis=10992.857m, share=0.181, loss=2902.786m, gain=941.916m


C:\Users\aliso\AppData\Local\Temp\ipykernel_18052\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)
C:\Users\aliso\AppData\Local\Temp\ipykernel_18052\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=Fals

[2016][AUT] +mis=13324.155m, -mis=13324.155m, share=0.326, loss=3795.773m, gain=1714.144m
[2016][BEL] +mis=44094.909m, -mis=44094.909m, share=0.591, loss=13165.851m, gain=3190.509m


C:\Users\aliso\AppData\Local\Temp\ipykernel_18052\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)
C:\Users\aliso\AppData\Local\Temp\ipykernel_18052\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=Fals

...

C:\Users\aliso\AppData\Local\Temp\ipykernel_18052\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)
C:\Users\aliso\AppData\Local\Temp\ipykernel_18052\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=Fals

[2016][SVN] +mis=34.964m, -mis=34.964m, share=0.081, loss=5.400m, gain=4.386m
[2016][SWE] +mis=31953.724m, -mis=31953.724m, share=0.326, loss=9102.947m, gain=4110.827m


C:\Users\aliso\AppData\Local\Temp\ipykernel_18052\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)
C:\Users\aliso\AppData\Local\Temp\ipykernel_18052\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=Fals

[2016][USA] +mis=193759.227m, -mis=193759.227m, share=0.324, loss=53036.275m, gain=9088.368m
[2016][ZAF] +mis=871.155m, -mis=871.155m, share=0.073, loss=214.731m, gain=46.199m


C:\Users\aliso\AppData\Local\Temp\ipykernel_18052\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


## Step 6. Checking the datasets

### Step 6.1. Checking the countries

In [18]:
# Open the CSV file f'{output_tables}/Scaling_Mario/SOTJ_sample_2016.csv'
sotj_2016_countries = pd.read_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_countries_2016.csv')

#Show me 2 decimals
pd.options.display.float_format = '{:,.2f}'.format
sotj_2016_countries[sotj_2016_countries['iso_partner'] == 'FRA']

,iso_partner,negative_misalignment,positive_misalignment,theoretical_profit,reported_profit,partner_jurisdiction,etr_average_corrected,cit,tax_revenue_current_usd,gvt_health_expenditure,region_tjn,ukt,oecd,oecd_oct,nld_oct,tax_revenue_loss,tax_revenue_gain,tax_revenue_loss_pct_of_gvt_health_expenditure,tax_revenue_loss_pct_of_total_tax_revenues,tax_revenue_loss_caused_pct_of_total,tax_revenue_loss_caused_usd,tax_revenue_loss_suffered_pct_of_total
54,FRA,"19,589.60",0.00,"134,300.32","112,603.82",France,0.19,0.34,"570,384,960,215.54","212,992,618,347.21",Europe,0.00,1.00,0.00,0.00,"6,745.35",0.00,0.03,0.01,0.00,0.00,0.05


### Step 6.2. Checking the aggregate results

In [19]:
sotj_2016_aggregate = pd.read_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_aggregate_results_2016.csv')
sotj_2016_aggregate


,year,total_positive_misalignment,total_negative_misalignment,total_profits,misaligned_of_total_profits,total_tax_revenue_loss,total_tax_revenue_gain,average_tax_revenue_loss_pct_of_gvt_health_expenditure,average_tax_revenue_loss_pct_of_total_tax_revenues
0,2016,"530,615.31","530,615.31","2,661,709.50",0.20,"147,413.25","40,946.06",0.09,0.01
